#### Purpose of this notebook

- I am generally interested in getting a handle on how many of my small variants have a titratable response
- Documenting the metabolic processes they belong to is also going to be critical
    - On this note, using the clusters themselves will be easier to make the case for enrichment
- Determining if there are degrees of size regulation (in terms of the slope) especially among tRNA synthetases

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr
import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances

from sklearn.metrics import silhouette_score
import scipy.stats
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering

import scanpy as sc
import anndata
import scipy as sp
import scipy.sparse
import dask.array as da
# from igraph.drawing.text import TextDrawer
from tslearn.neighbors import KNeighborsTimeSeries
from tslearn.metrics import cdist_soft_dtw_normalized,cdist_soft_dtw
import networkx as nx
import igraph as ig
import leidenalg
import umap
from scanpy.plotting.palettes import default_20,vega_20_scanpy
from matplotlib import pyplot as plt
from matplotlib import cm
import ast
from tslearn.barycenters import softdtw_barycenter,dtw_barycenter_averaging,euclidean_barycenter
import xarray as xr

import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import re
import goatools
import goatools.base
from goatools.base import download_go_basic_obo

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.semantic import semantic_similarity
from goatools.semantic import TermCounts, get_info_content

from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.go_enrichment import GOEnrichmentStudy

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

In [ ]:
def highlight_regex(an_df,regex,column_label="Gene",color_all_mode="red",highlight="all",pallete=vega_20_scanpy,match_mode="regex",fontsize=18):
    ## highlight ("all","each") will either render all matches as a single color ("all") or each match type (e.g. gene name) as its own color ("each") acoording to the pallete
    ## match_mode ("regex","exact") will either perform a re.search ("regex") or exact match ("exact") for its search
    ## regex may be either (1) a regex string, (2) an exact match string or (3) a list (when in exact match mode) of exact matches to search
    
    
    highlight_df = copy.deepcopy(an_df)
    regex_title = regex
    
    if match_mode == "regex":
        re_pattern = re.compile(regex)
        regex_mask = highlight_df.obs[column_label].apply(lambda x: re.search(re_pattern,x) != None)
        regex_name_list = sorted(highlight_df.obs[regex_mask][column_label].unique().tolist())
    
    elif match_mode == "exact":
        if type(regex) == list:
            regex_mask = highlight_df.obs[column_label].apply(lambda x: any([x==regex_val for regex_val in regex]))
            regex_title = ",".join(regex_title)
        else:
            regex_mask = highlight_df.obs[column_label].apply(lambda x: x==regex)
        regex_name_list = sorted(highlight_df.obs[regex_mask][column_label].unique().tolist())
    
    if highlight == "all":
        highlight_df.obs[column_label + " | Regex: " + regex_title] = regex_mask.astype("category")
        fig = sc.pl.umap(highlight_df, title=column_label + " | Regex: " + regex_title, color=column_label + " | Regex: " + regex_title,
                     groups=[True], show=False, legend_loc='right margin', add_outline=False, size=50, return_fig=True,
                     palette = {True:color_all_mode,False:"lightgrey"});
    elif highlight == "each":
        highlight_df.obs[column_label + " | Regex: " + regex_title] = highlight_df.obs[column_label].apply(lambda x: x if x in regex_name_list else np.NaN).astype("category")
        fig = sc.pl.umap(highlight_df, title=column_label + " | Regex: " + regex_title, color=column_label + " | Regex: " + regex_title, groups=regex_name_list, show=False, legend_loc='right margin', \
                         add_outline=False, size=50, return_fig=True,palette=pallete);
    
    
    axes = fig.get_axes()
    for ax in axes:
        ax.set_title(ax.get_title(), fontsize=fontsize)
        ax.set_ylabel(ax.get_ylabel(), fontsize=fontsize)
        ax.set_xlabel(ax.get_xlabel(), fontsize=fontsize)
        
    return fig
    
def highlight_cluster(an_df,cluster_level,cluster_label,resolution_list,color="purple",fontsize=18):
    
    fig = highlight_regex(an_df,cluster_label,column_label=cluster_level,color_all_mode=color,highlight="all",match_mode="exact",fontsize=fontsize);
    
    cluster_level_index = int(cluster_level[1:])
    
    axes = fig.get_axes()
    for i,ax in enumerate(axes):
        if i%2==0:
            ax.set_title("Leiden Resolution=" + str(resolution_list[cluster_level_index]) + " | " + str(cluster_label), fontsize=fontsize)
            
    return fig

def search_go(ns2assoc, obodag, inv_gene_to_id, go_term):
    namespace_abbv = {"biological_process":"BP","molecular_function":"MF","cellular_component":"CC"}
    
    print("Searching for " + str(obodag[go_term].name))
    namespace = namespace_abbv[obodag[go_term].namespace]
    child_goterms = list(obodag[go_term].get_all_children())
    gene_list = [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if go_term in val]
    for child_goterm in child_goterms:    
        gene_list += [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if child_goterm in val]
    gene_list = sorted(list(set(gene_list)))
    return gene_list

def highlight_go(an_df, ns2assoc, obodag, inv_gene_to_id, go_term, go_mode = "all", color="red", fontsize=18):
    
    gene_list = search_go(ns2assoc, obodag, inv_gene_to_id, go_term)
    if go_mode == "all":
        fig = highlight_regex(an_df,gene_list,column_label="Gene",color_all_mode=color,highlight="all",match_mode="exact",fontsize=fontsize)
    elif go_mode == "each":
        fig = highlight_regex(an_df,gene_list,column_label="Gene",highlight="each",match_mode="exact",fontsize=fontsize)
        
    plt.title(go_term)
    return fig

def get_enriched_GO_terms(background_gene_list,gene_list,obodag,objanno,ns2assoc,pval=0.05,GO_type="BP"):
    
    gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
    synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
    gene_to_id.update(synonym_dict)

    #background gene set

    all_genes_uniprot = [gene_to_id[item] for item in background_gene_list if item in gene_to_id.keys()]
    selected_genes_uniprot = [gene_to_id[item] for item in gene_list if item in gene_to_id.keys()]
    
    print(len(all_genes_uniprot))
    print(len(selected_genes_uniprot))
    
    goeaobj = GOEnrichmentStudy(
            all_genes_uniprot, # List of mouse protein-coding genes
            ns2assoc[GO_type], # geneid/GO associations
            obodag, # Ontologies
            propagate_counts = True,
            alpha = pval, # default significance cut-off
            methods=["fdr_bh"]); # defult multipletest correction method

    goea_results_all = goeaobj.run_study(selected_genes_uniprot, prt=None)
    goea_quiet_sig = [r for r in goea_results_all if r.p_fdr_bh < pval]
    goea_quiet_enriched = [r for r in goea_quiet_sig if r.enrichment == "e"]
    return goea_quiet_enriched

def pick_exemplar(go1,go2,termcounts,obodag,info_thr,pval_factor=2.):
    
    info_1_low = get_info_content(go1.GO, termcounts)<info_thr
    info_2_low = get_info_content(go2.GO, termcounts)<info_thr
    if info_1_low and not info_2_low:
        return go2
    elif info_2_low and not info_1_low:
        return go1
    elif info_2_low and info_1_low:
        return go1
    
    pval_ratio = go1.p_fdr_bh/go2.p_fdr_bh
    
    if pval_ratio > pval_factor:
        return go2
    elif pval_ratio < (1./pval_factor):
        return go1
    
    go1_parents = list(obodag[go1.GO].get_all_parents())
    go2_parents = list(obodag[go2.GO].get_all_parents())
    
    if go2.GO in go1_parents:
        return go2
    
    elif go1.GO in go2_parents:
        return go1
    
    return go1

def get_filtered_go_terms(obodag,objanno,goea_list,sim_thr = 0.05,info_thr = 1.,GO_type="BP"):
    
    termcounts = TermCounts(obodag, objanno.get_ns2assc()[GO_type])

    go_term_list = [item.GO for item in goea_list]
    sim_arr = np.zeros((len(go_term_list),len(go_term_list)))
    for i in range(len(go_term_list)):
        for j in range(len(go_term_list)):
            sim_arr[i,j] = semantic_similarity(go_term_list[i], go_term_list[j], obodag)
    np.fill_diagonal(sim_arr, 0.)
    
    working_group_idx = 0
    grouped_terms = {}
    group_exemplars = {}
    go_term_indices = list(range(len(go_term_list)))

    while len(go_term_indices)>0:
        i = go_term_indices[0]
        most_sim_arg = np.argmax(sim_arr[i])
        sim_score = sim_arr[i,most_sim_arg]
        if sim_score > sim_thr:
            if len(grouped_terms)>0:
                in_other_group_keys = [key for key,val in grouped_terms.items() if most_sim_arg in val]
                if len(in_other_group_keys)==1:
                    other_group_idx = in_other_group_keys[0]
                    grouped_terms[other_group_idx] = grouped_terms[other_group_idx] + [i]
                    group_exemplars[other_group_idx] = pick_exemplar(group_exemplars[other_group_idx],goea_list[i],termcounts,obodag,info_thr)
                else:        
                    grouped_terms[working_group_idx] = [i,most_sim_arg]
                    group_exemplars[working_group_idx] = pick_exemplar(goea_list[i],goea_list[most_sim_arg],termcounts,obodag,info_thr)
                    working_group_idx += 1
                    go_term_indices.remove(most_sim_arg)
            else:
                grouped_terms[working_group_idx] = [i,most_sim_arg]
                group_exemplars[working_group_idx] = pick_exemplar(goea_list[i],goea_list[most_sim_arg],termcounts,obodag,info_thr)
                working_group_idx += 1
                go_term_indices.remove(most_sim_arg)
        go_term_indices.remove(i)
    
    group_exemplars = list(group_exemplars.values())
    
    return group_exemplars


def get_GO_assign_dict(selected_goea,cluster_genes_uniprot):
    all_study_items = copy.copy(cluster_genes_uniprot)
    depth_list = sorted(set([item.depth for item in selected_goea]))[::-1]
    assign_dict = {}
    for depth in depth_list:
        go_terms_at_level = [item for item in selected_goea if item.depth == depth]
        for go_term in go_terms_at_level:
            study_item_list = list(go_term.study_items)
            for study_item in study_item_list:
                if study_item in all_study_items:
                    assign_dict[study_item] = go_term.name
                    all_study_items.remove(study_item)

    for remaining_item in all_study_items:
        assign_dict[remaining_item] = "Unassigned"
        
    return assign_dict

In [ ]:
short_expt_strong_effect_threshold = 1.5
long_expt_strong_effect_threshold = 1.5

##override with short for now
short_expt_rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-02-26_lDE20_10Hour_Analysis"
long_expt_rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-01-02_lDE23_32Hour_Analysis"
short_expt_z_score_thr_dir = short_expt_rootpath + "/Z_Score_Thr_" + str(short_expt_strong_effect_threshold) + "_N_Neighbors_10"
long_expt_z_score_thr_dir = long_expt_rootpath + "/Z_Score_Thr_" + str(long_expt_strong_effect_threshold) + "_N_Neighbors_10"

short_expt_clustree_pandas_df = pd.read_pickle(short_expt_z_score_thr_dir+"/Pandas_Dataframe.pkl")
short_expt_an_df_clustree_r_compat = anndata.read_h5ad(short_expt_z_score_thr_dir+"/AnnData.h5ad")
long_expt_clustree_pandas_df = pd.read_pickle(long_expt_z_score_thr_dir+"/Pandas_Dataframe.pkl")
long_expt_an_df_clustree_r_compat = anndata.read_h5ad(long_expt_z_score_thr_dir+"/AnnData.h5ad")

In [ ]:
long_expt_length_width_df = long_expt_clustree_pandas_df[['Kernel Trace: Delta: Volume','Kernel Trace: Division: major_axis_length',\
                                                          'Kernel Trace: minor_axis_length',"Kernel Trace: Instantaneous Growth Rate: Volume"]]
short_expt_length_width_df = short_expt_clustree_pandas_df[['Kernel Trace: Delta: Volume','Kernel Trace: Division: major_axis_length',\
                                                          'Kernel Trace: minor_axis_length',"Kernel Trace: Instantaneous Growth Rate: Volume"]]

long_expt_growth_df_t0 = long_expt_length_width_df.apply(lambda x: [column[0] for _,column in x.iteritems()])
short_expt_growth_df_t0 = short_expt_length_width_df.apply(lambda x: [column[0] for _,column in x.iteritems()])

long_expt_growth_df_t_24hr_post = long_expt_length_width_df.apply(lambda x: [column[31] for _,column in x.iteritems()])
short_expt_growth_df_t_8hr_post = short_expt_length_width_df.apply(lambda x: [column[19] for _,column in x.iteritems()])

short_expt_growth_df_t_8hr_post["Gene"] = short_expt_clustree_pandas_df["Gene"]
short_expt_growth_df_t_8hr_post["TargetID"] = short_expt_clustree_pandas_df["TargetID"]

In [ ]:
# Get ontologies
obo_fname = download_go_basic_obo()

# Get ecoli association file (ecocyc)
gaf_handle = goatools.base.http_get("http://current.geneontology.org/annotations/ecocyc.gaf.gz",fout=long_expt_rootpath + "/ecocyc.gaf.gz")
gaf_fname = goatools.base.gunzip(long_expt_rootpath + "/ecocyc.gaf.gz")

## Getting ontologies and other nonesense

obodag = GODag(obo_fname)
objanno = GafReader(gaf_fname)
ns2assoc = objanno.get_ns2assc()

gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
inv_gene_to_id = {assoc.DB_ID:assoc.DB_Symbol for assoc in objanno.associations}
synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
gene_to_id.update(synonym_dict)

### Step 1: Discovery
- Genes which exhibit this phenotype can be readily identified from the clustering
- Ignoring curation for now, find the major biosynthetic groups and plot them here

In [ ]:
### discovery will be cluster-based in the near future
### fill this in after curation

In [ ]:
short_expt_clustree_pandas_df[short_expt_clustree_pandas_df["L0"]=="8"]["Gene"]

In [ ]:
cluster_df = short_expt_growth_df_t_8hr_post.loc[short_expt_clustree_pandas_df[(short_expt_clustree_pandas_df["L0"].isin(["1","3","8"]))].index]
not_cluster_df = short_expt_growth_df_t_8hr_post.loc[short_expt_clustree_pandas_df[~(short_expt_clustree_pandas_df["L0"].isin(["1","3","8"]))].index]

In [ ]:
plt.xlim(0,1.4)
plt.ylim(3,14)
plt.yscale("log", base=2)

plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
plt.scatter(not_cluster_df["Kernel Trace: Instantaneous Growth Rate: Volume"],not_cluster_df["Kernel Trace: Division: major_axis_length"],s=2)
plt.scatter(cluster_df["Kernel Trace: Instantaneous Growth Rate: Volume"],cluster_df["Kernel Trace: Division: major_axis_length"],s=2)
plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
plt.tight_layout()

In [ ]:
### come up with a strong enrichment criterion later
n_count_min = 5

gene,counts = np.unique(cluster_df["Gene"],return_counts=True)
genes_past_thr = gene[counts>=n_count_min]

In [ ]:
fatty_acid_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006633")

In [ ]:
genes_past_thr

In [ ]:
[gene_name for gene_name in genes_past_thr if gene_name in fatty_acid_biosyn]

### Step 2: Fitting Slopes
- Enumerate major biosynthetic groups
- fit slopes to non-polar/off-target examples

#### Fatty Acid Biosynthesis

In [ ]:
fatty_acid_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006633")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in fatty_acid_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
len(gene_list)

In [ ]:
fig = plt.figure(figsize=(8,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(3,4,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

### LPS Synthesis

In [ ]:
lps_biosyn_1 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009103")
lps_biosyn_2 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009244")
lps_biosyn_3 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009243")
lps_biosyn_4 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0019294")
lps_biosyn = lps_biosyn_1 + lps_biosyn_2 + lps_biosyn_3 + lps_biosyn_4 + ["msbA","rfaD","rfaE","waaC","waaF"]
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in lps_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(8,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(3,4,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### tRNA Synthetases (including from all clusters since a couple not in above)

In [ ]:
trna_synthetases = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006418")
group_df = short_expt_growth_df_t_8hr_post[short_expt_growth_df_t_8hr_post["Gene"].isin(trna_synthetases)]
gene_list = group_df["Gene"].unique().tolist()

In [ ]:
fig = plt.figure(figsize=(9,9))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(5,5,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Coenzyme A Biosynthetic Process

In [ ]:
coa_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0015937")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in coa_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(9,3))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(1,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Glycolysis

In [ ]:
glycolysis = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006096")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in glycolysis]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(6,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(3,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Riboflavin Biosynthetic Process

In [ ]:
ribo_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009231")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in ribo_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(6,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(3,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

In [ ]:
isopr_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0008299")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in isopr_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(6,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(3,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### tRNA Processing

In [ ]:
trna_processing = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0008033")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in trna_processing]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(6,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(3,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Nucleotide Biosynthetic Process

In [ ]:
nuc_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009117")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in nuc_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(6,6))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(4,5,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Quorum

In [ ]:
quorum = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009372")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in quorum]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(3,3))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(1,1,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Division

In [ ]:
division = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0051301")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in division]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(9,3))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(1,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Shape

In [ ]:
shape = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0008360")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in shape]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(9,3))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(1,3,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

#### Redox

In [ ]:
redox = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0045454")
gene_list = [gene_name for gene_name in genes_past_thr if gene_name in redox]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

In [ ]:
fig = plt.figure(figsize=(3,3))

for i,gene_name in enumerate(gene_list):
    gene_df = group_df[group_df["Gene"]==gene_name]
    
    plt.subplot(1,1,i+1)
    plt.title(gene_name)
    plt.xlim(0,1.4)
    plt.ylim(3,14)
    plt.yscale("log", base=2)

    plt.yticks(ticks=[3,6,12],labels=["3","6","12"])
    plt.scatter(gene_df["Kernel Trace: Instantaneous Growth Rate: Volume"],gene_df["Kernel Trace: Division: major_axis_length"],s=2,color="black")
    # plt.ylabel(r"Length $ (\mu m) $",fontsize=16)
    # plt.xlabel(r"Growth Rate $ (1/hr) $",fontsize=16)

    plt.tick_params(axis='x', labelsize=16)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=16)  # Set x-tick label size
    
fig.supxlabel(r"Growth Rate $ (1/hr) $",fontsize=16)
fig.supylabel(r"Length $ (\mu m) $",fontsize=16)

plt.tight_layout()

### Fitting Slopes: tRNA Synthetases

In [ ]:
trna_synthetases = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006418")
group_df = short_expt_growth_df_t_8hr_post[short_expt_growth_df_t_8hr_post["Gene"].isin(trna_synthetases)]

In [ ]:
example_df = group_df[group_df["Gene"] == "thrS"]

In [ ]:
import sklearn as skl

In [ ]:
def fit_reg(df):
    reg_model = skl.linear_model.LinearRegression()
    reg_model.fit(df["Kernel Trace: Instantaneous Growth Rate: Volume"].values.reshape(-1,1),\
                  np.log2(df["Kernel Trace: Division: major_axis_length"].values))
    slope,intercept = reg_model.coef_[0],reg_model.intercept_
    r2 = reg_model.score(df["Kernel Trace: Instantaneous Growth Rate: Volume"].values.reshape(-1,1),\
                  np.log2(df["Kernel Trace: Division: major_axis_length"].values))
    output = pd.Series({"Slope":slope,"Intercept":intercept, "R2":r2})
    return output

In [ ]:
tRNA_slopes = group_df.groupby("Gene").apply(lambda x: fit_reg(x))

In [ ]:
plt.hist(tRNA_slopes["Slope"],range=(0,2),histtype='step')

In [ ]:
tRNA_slopes[tRNA_slopes["Slope"]<0.7]

In [ ]:
tRNA_slopes[tRNA_slopes["Slope"]>1.2]

#### Interesting Observations
- The extreme slopes (on both sides) are composed of positivly charged side chains
- His is a very weak stimulator of the stringent response in general
- His is notable as it is connected to nucleotide synthesis by PRPP
- Possible that the abundance of nucleotides relieves the induction somewhat by increasing availability of precursors for all other amino acids?
- Also possible that histidine tRNA less able to stimulate stringent response, they are structurally a bit distinct

### Fitting Slopes: All Examples (in cluster)

In [ ]:
min_r2 = 0.7
min_points = 8

gene,counts = np.unique(cluster_df["Gene"],return_counts=True)
genes_past_thr = gene[counts>=n_count_min]
filtered_cluster_df = cluster_df[cluster_df["Gene"].isin(genes_past_thr)]

all_slopes = filtered_cluster_df.groupby("Gene").apply(lambda x: fit_reg(x))
all_slopes = all_slopes[all_slopes["R2"]>=min_r2]

In [ ]:
plt.hist(all_slopes["Slope"],range=(-2,4),bins=20)

#### Strong Inducers
- asnS
- lysS
- pgm
- tyrS
- ubiX
- ymfE (possibly)

In [ ]:
all_slopes[all_slopes["Slope"]>1.25].index.tolist()

#### Weak Inducers
- hisS
- coaA (other coa)
- fbaA
- glyVWXY
- ribA (other rib)

In [ ]:
all_slopes[all_slopes["Slope"]<0.6]

### Notes

- Clearly not done with this
- Try to clean up later in the day
- Main results to focus on:
    - Hierarchy of size scalings for tRNA starvation
    - Hierarchy of size scalings disrupting different pathways (just select a few based on metabolic role)
- If I have those two results in hand; I don't need to worry about the rest of it...go from first principles (what metabolite is being depleted)...the nice thing about the tRNA synthetases is that they are specific...try to find similar things
- Do not necessarily need clustering input here...can go with biological intuition
- Once those two outputs are present in clean form..

In [ ]:
#### tRNA Synthetases (including from all clusters since a couple not in above)

In [ ]:
fatty_acid_biosyn = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006633")
gene_list_1 = [gene_name for gene_name in genes_past_thr if gene_name in fatty_acid_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

lps_biosyn_1 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009103")
lps_biosyn_2 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009244")
lps_biosyn_3 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0009243")
lps_biosyn_4 = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0019294")
lps_biosyn = lps_biosyn_1 + lps_biosyn_2 + lps_biosyn_3 + lps_biosyn_4 + ["msbA","rfaD","rfaE"]
gene_list_2 = [gene_name for gene_name in genes_past_thr if gene_name in lps_biosyn]
group_df = cluster_df[cluster_df["Gene"].isin(gene_list)]

trna_synthetases = search_go(ns2assoc, obodag, inv_gene_to_id, "GO:0006418")
group_df = short_expt_growth_df_t_8hr_post[short_expt_growth_df_t_8hr_post["Gene"].isin(trna_synthetases)]
gene_list_3 = group_df["Gene"].unique().tolist()

In [ ]:
sorted(list(set(genes_past_thr) - (set(gene_list_1)|set(gene_list_2)|set(gene_list_3))))